# Excecise 1 - Bias Detection and Mitigation

- Understand three common bias types in ML: historical, representation, and measurement.
- Use simple statistics and sampling techniques (SMOTE, undersampling) to explore and mitigate bias in datasets.
- Practice evaluating classifier behavior across groups and thresholds.

**Notebook Structure:**
1. Data exploration and statistical checks (detect historical & representation bias).
2. Addressing label imbalance with SMOTE and undersampling (measurement/label bias mitigation).
3. Applying approaches to a real healthcare dataset and tuning thresholds.

### Bias Detection — Key Concepts
- Historical bias: arises from past social or institutional practices reflected in the data (e.g., systemic under-treatment). Detect by inspecting outcome distributions across groups and over time.
- Representation bias: occurs when certain groups are under- or over-represented in the data relative to the population of interest. Detect with simple counts and group proportions.
- Measurement bias: when features or labels are measured differently across groups (e.g., proxy labels). Detect by checking for different data-quality or proxy patterns per group, and by validating label definitions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support


from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from fairlearn.datasets import fetch_diabetes_hospital


def get_class_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if y_true.size == 0 or y_pred.size == 0:
        print("Empty input: no samples to evaluate.")
        return

    if y_true.shape[0] != y_pred.shape[0]:
        print(f"Length mismatch: y_true={y_true.shape[0]}, y_pred={y_pred.shape[0]}")
        return

    if np.unique(y_true).size < 2:
        print("ROC-AUC: undefined (only one class present in y_true)")
    else:
        print(f"ROC-AUC: {roc_auc_score(y_true, y_pred):.3f}")

    print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.3f}")
    print(f"Recall: {recall_score(y_true, y_pred, zero_division=0):.3f}")
    print(f"F1: {f1_score(y_true, y_pred, zero_division=0):.3f}")

def clean_hospital_data(X):
    data = X.copy()

    data["change"] = data["change"].apply(lambda x: True if x == "Ch" else False)
    data["diabetesMed"] = data["diabetesMed"].apply(lambda x: True if x == "Yes" else False)
    data["medicaid"] = data["medicaid"].apply(lambda x: True if x == "True" else False)
    data["medicare"] = data["medicare"].apply(lambda x: True if x == "True" else False)
    data["had_emergency"] = data["had_emergency"].apply(lambda x: True if x == "True" else False)
    data["had_inpatient_days"] = data["had_inpatient_days"].apply(lambda x: True if x == "True" else False)
    data["had_outpatient_days"] = data["had_outpatient_days"].apply(lambda x: True if x == "True" else False)

    categorical_cols = ['race',
    'gender',
    'age',
    'discharge_disposition_id',
    'admission_source_id',
    'medical_specialty',
    'primary_diagnosis',
    'insulin']

    df_dummy = pd.get_dummies(data[categorical_cols])

    data = pd.concat([data, df_dummy], axis = 1)
    data.drop(categorical_cols, axis = 1, inplace=True)

    return data

## 1. Data exploration: detect historical & representation bias

This section demonstrates quick, actionable checks to reveal historical and representation bias in datasets.
- Historical bias: look for outcome differences across groups that may reflect past practices.
- Representation bias: check group counts and proportions to see under- or over-representation.

We use a small healthcare dataset as an example. The dataset contains 10 years of clinical admission records from 130 US hospitals. Features include demographics, diagnoses, medications, visits, payer information, and readmission labels.

Source: [Diabetes 130-Hospitals Dataset](https://fairlearn.org/main/user_guide/datasets/diabetes_hospital_data.html)

Follow the code cells below to compute basic counts, group breakdowns, and simple visual checks that highlight potential biases.

## Fetching the data

**Part A - Data load & initial checks**: load the diabetes dataset.
- Bias focus: representation & historical bias detection (prepare to run quick counts and group checks).

In [ ]:
data = fetch_diabetes_hospital(as_frame=True)
med_data = data.data.copy()
med_data.drop(["max_glu_serum", "A1Cresult", "readmitted"], axis = 1, inplace=True)
y_true = data.target

In [ ]:
med_data.iloc[1]

## Simple stats
First, check the counts of the target variable to identify potential imbalance.

Let's inspect whether readmission within 30 days is under-represented in the data.

**Check - Label distribution**: compute counts of the target label.
- Bias focus:  Class imbalance (measurement/label issues)
- Historical bias if outcome rates differ across groups or over time, which may reflect past practices.

In [ ]:
y_true.value_counts()

**Check - Representation by race**:
- Let's assess race to see whether representation is uneven. For example, are Caucasian records more numerous than other groups?
- Bias focus: representation bias - note under/over-represented groups; consider downstream effects on model fairness.

In [ ]:
df = med_data["race"].value_counts().reset_index()
df["percent"] = df["count"] / df["count"].sum() * 100

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(df["race"], df["percent"])
ax.set_title("Percentage by race")
ax.tick_params(axis="x", labelrotation=45)

plt.tight_layout()
plt.show()

Let's examine gender distribution within different races to check balance.

**Check - Within-group composition**: compute gender mix per race.
- Bias focus: representation imbalance within a subgroup can amplify downstream disparities.

In [ ]:
(pd.crosstab(med_data["race"], med_data["gender"], normalize="index") * 100).round(1)

For example, the African American subgroup shows a higher proportion of female patients in this dataset.

**Check - Outcome/feature distribution by subgroup**: compare hospital stay stats by age within the Caucasian subgroup.

In [ ]:
df = med_data[med_data["race"] == "Caucasian"].groupby("age")["time_in_hospital"].describe()
df.style.format({
    "count": "{:.0f}",
    "mean": "{:.1f}", 
    "std": "{:.1f}", 
    "min": "{:.0f}", 
    "25%": "{:.0f}", 
    "50%": "{:.0f}", 
    "75%": "{:.0f}", 
    "max": "{:.0f}"})

## 2. Addressing label imbalance

Label imbalance can cause poor performance for minority labels and can be a form of measurement bias when the label is a proxy measured unevenly across groups.
This section shows how to use sampling techniques (SMOTE and combined under/over-sampling) to rebalance the training data and evaluate the effect on classifier metrics.

Note: rebalancing can improve predictive performance but may not correct underlying measurement issues - always inspect feature and label quality across groups.

### Baseline on synthetic imbalanced data
- Train a decision tree on a heavily imbalanced synthetic dataset.
- Bias focus: shows how class imbalance hurts minority-label performance (measurement/label bias).

In [ ]:
# define a inbalaced dataset
X, y = make_classification(n_samples=20000, n_features=10, n_informative=2, n_redundant=3,
	n_clusters_per_class=2, weights=[0.99], flip_y=0, random_state=1)

print(np.unique(y, return_counts = True))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
print(f"Training set positive count: {np.sum(y_train)}")
print(f"Test set positive count: {np.sum(y_test)}")

# define model
model = DecisionTreeClassifier()

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

get_class_metrics(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y))
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

### SMOTE oversampling

Apply SMOTE to the training data and compare metrics to the baseline.

Synthetic Minority Over-sampling Technique (SMOTE) generates synthetic minority-class examples by interpolating between existing minority samples in feature space. This increases minority class support without simple duplication.

When to use: useful when the minority class is under-represented and simple replication would lead to overfitting. SMOTE is often combined with undersampling of the majority class for better balance.

Caveats: synthetic samples may be unrealistic in some domains, can amplify noisy minority examples, and may shift class boundaries. Always validate improvements on held-out data and inspect per-group effects.

In [ ]:
# define pipeline
steps = [('over', SMOTE(random_state=42)), ('model', DecisionTreeClassifier())] # use SMOTE oversampling
pipeline = Pipeline(steps=steps)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

get_class_metrics(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y))
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

It is believed that SMOTE performs better when combined with undersampling of the majority class, such as random undersampling.

We can achieve this by simply adding a RandomUnderSampler step to the Pipeline.

### Combined SMOTE + undersampling
- Try SMOTE with random undersampling of the majority class.
- Bias focus: balanced sampling strategies may improve minority detection while limiting synthetic oversampling risks.

In [ ]:
# define pipeline
model = DecisionTreeClassifier()
over = SMOTE(sampling_strategy=0.1)
under = RandomUnderSampler(sampling_strategy=0.5)
steps = [('over', over), ('under', under), ('model', model)]
pipeline = Pipeline(steps=steps)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

get_class_metrics(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y))
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

## 3. Apply to the diabetes dataset: group-aware evaluation & threshold tuning

Here we apply the previous ideas to the real diabetes dataset. We transform categorical features, fit a classifier, and demonstrate evaluation techniques including probability threshold tuning.
- Goal: evaluate whether rebalancing or threshold changes help the minority label and whether performance differs across demographic groups.
- Tip: when possible, compute per-group metrics (precision/recall/F1) to detect disparate performance.

### Baseline evaluation
- Bias focus: initial performance may be poor for the minority label (measurement bias) and may differ across groups (historical/representation bias).

In [ ]:
data = fetch_diabetes_hospital(as_frame=True)
X = data.data.copy()
X.drop(["max_glu_serum", "A1Cresult", "readmitted", "readmit_binary"], axis = 1, inplace=True)
y = data.target

X["change"] = X["change"].apply(lambda x: True if x == "Ch" else False)
X["diabetesMed"] = X["diabetesMed"].apply(lambda x: True if x == "Yes" else False)
X["medicaid"] = X["medicaid"].apply(lambda x: True if x == "True" else False)
X["medicare"] = X["medicare"].apply(lambda x: True if x == "True" else False)
X["had_emergency"] = X["had_emergency"].apply(lambda x: True if x == "True" else False)
X["had_inpatient_days"] = X["had_inpatient_days"].apply(lambda x: True if x == "True" else False)
X["had_outpatient_days"] = X["had_outpatient_days"].apply(lambda x: True if x == "True" else False)

categorical_cols = ['race',
 'gender',
 'age',
 'discharge_disposition_id',
 'admission_source_id',
 'medical_specialty',
 'primary_diagnosis',
 'insulin']

df_dummy = pd.get_dummies(X[categorical_cols])

X = pd.concat([X, df_dummy], axis = 1)
X.drop(categorical_cols, axis = 1, inplace=True)

print(np.unique(y, return_counts = True))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# define model
# model = RandomForestClassifier(n_estimators=100, random_state=42)
model = RandomForestClassifier(n_estimators=10, max_depth=25, criterion="gini", min_samples_split=10, random_state=42)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

get_class_metrics(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y))
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

### Threshold tuning and per-threshold diagnostics:
- Compute predicted probabilities, sweep thresholds, and select one by F1.
- Bias focus: thresholding can change trade-offs differently for groups; consider per-group thresholds or cost-sensitive decisions when fairness constraints require it.

In [ ]:
y_probs = model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.0, 0.76, 0.01)
f1_scores = []
roc_scores = []

for t in thresholds:
    y_pred = (y_probs >= t).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
    f1_scores.append(f1)
    roc_scores.append(roc_auc_score(y_test, y_pred))

# Find best threshold by highest F1
best_threshold = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)
best_roc = roc_scores[np.argmax(f1_scores)]

print(f"Best Threshold: {best_threshold:.2f}")
print(f"Best F1 Score: {best_f1:.3f}")
print(f"ROC-AUC Score at best threshold: {best_roc:.3f}")

y_pred_final = (y_probs >= best_threshold).astype(int)

cm = confusion_matrix(y_test, y_pred_final)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y))
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

In [ ]:
precisions = []
recalls = []
f1s = []

for t in thresholds:
    y_pred = (y_probs >= t).astype(int)
    precisions.append(precision_score(y_test, y_pred, zero_division=0))
    recalls.append(recall_score(y_test, y_pred, zero_division=0))
    f1s.append(f1_score(y_test, y_pred, zero_division=0))

plt.figure(figsize=(10,6))
plt.plot(thresholds, precisions, label='Precision')
plt.plot(thresholds, recalls, label='Recall')
plt.plot(thresholds, f1s, label='F1 Score')
plt.axvline(best_threshold, color='gray', linestyle='--', label=f'Best Threshold = {best_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Threshold vs Precision, Recall, F1')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Group-wise performance
def print_metrics_for_group(group_name, X_test, y_true, y_pred):
    idx = X_test[group_name] == 1
    print(f"Group: {group_name}")
    get_class_metrics(y_true[idx], y_pred[idx])
    print("-" * 30)

group_names = ['gender_Female', 'gender_Male', 'gender_Unknown/Invalid']
for group in group_names:
    print_metrics_for_group(group, X_test, y_test, y_pred_final)

In [ ]:
# Race-wise performance
group_names = ['race_AfricanAmerican', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Other', 'race_Unknown']
for group in group_names:
    print_metrics_for_group(group, X_test, y_test, y_pred_final)

### Undersampling on real data

- Use `RandomUnderSampler` as a pipeline step so resampling occurs only on training folds.
- Bias focus: undersampling may preserve real samples but can drop important majority-group variation; weigh trade-offs and compute per-group metrics after training.

In [ ]:
pipeline_under = Pipeline([
    ('under', RandomUnderSampler(sampling_strategy=0.5, random_state=42)),
    ('model', RandomForestClassifier(n_estimators=10, random_state=42))
])

pipeline_under.fit(X_train, y_train)
y_pred_under = pipeline_under.predict(X_test)

print('Undersampling pipeline results:')
get_class_metrics(y_test, y_pred_under)

cm = confusion_matrix(y_test, y_pred_under)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y))
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix - Undersampling pipeline')
plt.show()